In [1]:
!git clone https://github.com/MaurizioFD/RecSys_Course_AT_PoliMi.git


Cloning into 'RecSys_Course_AT_PoliMi'...
Updating files:  82% (165/199)
Updating files:  83% (166/199)
Updating files:  84% (168/199)
Updating files:  85% (170/199)
Updating files:  86% (172/199)
Updating files:  87% (174/199)
Updating files:  88% (176/199)
Updating files:  89% (178/199)
Updating files:  90% (180/199)
Updating files:  91% (182/199)
Updating files:  92% (184/199)
Updating files:  93% (186/199)
Updating files:  94% (188/199)
Updating files:  95% (190/199)
Updating files:  96% (192/199)
Updating files:  97% (194/199)
Updating files:  98% (196/199)
Updating files:  99% (198/199)
Updating files: 100% (199/199)
Updating files: 100% (199/199), done.


In [1]:
%cd RecSys_Course_AT_PoliMi
!pip install -r requirements.txt
!pip install pandas scipy


C:\Users\VOLKAN MAZLUM\RecSys_Course_AT_PoliMi

DEPRECATION: pyodbc 4.0.0-unsupported has a non-standard version number. pip 24.0 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pyodbc or contact the author to suggest that they release a version with a conforming version number. Discussion can be found at https://github.com/pypa/pip/issues/12063

[notice] A new release of pip is available: 23.3.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


DEPRECATION: pyodbc 4.0.0-unsupported has a non-standard version number. pip 24.0 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pyodbc or contact the author to suggest that they release a version with a conforming version number. Discussion can be found at https://github.com/pypa/pip/issues/12063

[notice] A new release of pip is available: 23.3.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
import scipy.sparse as sp
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from Evaluation.Evaluator import EvaluatorHoldout
from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender
from Recommenders.KNN.ItemKNNCustomSimilarityRecommender import ItemKNNCustomSimilarityRecommender

# Load datasets
data_train = pd.read_csv("C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\data_train.csv")
icm_metadata = pd.read_csv("C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\data_ICM_metadata.csv")
target_users = pd.read_csv("C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\data_target_users_test.csv")

# Process user-item interaction data
column_names = ['user_id', 'item_id', 'interaction']
dataframe = data_train.rename(columns={data_train.columns[0]: 'user_id', 
                                       data_train.columns[1]: 'item_id', 
                                       data_train.columns[2]: 'interaction'})

URM_all = sp.coo_matrix(
    (dataframe['interaction'].values, 
     (dataframe['user_id'].values, dataframe['item_id'].values))
)

# Process item-content metadata (ICM)
icm_column_names = ['item_id', 'feature_id', 'value']
icm_df = icm_metadata.rename(columns={icm_metadata.columns[0]: 'item_id', 
                                       icm_metadata.columns[1]: 'feature_id', 
                                       icm_metadata.columns[2]: 'value'})

ICM_all = sp.coo_matrix(
    (icm_df['value'].values, 
     (icm_df['item_id'].values, icm_df['feature_id'].values))
)



In [6]:

# Split data into train, validation, and test
URM_train_validation, URM_test = split_train_in_two_percentage_global_sample(URM_all, train_percentage=0.8)
URM_train, URM_validation = split_train_in_two_percentage_global_sample(URM_train_validation, train_percentage=0.8)

# Set up evaluators
evaluator_validation = EvaluatorHoldout(URM_validation, cutoff_list=[10])
evaluator_test = EvaluatorHoldout(URM_test, cutoff_list=[10])


EvaluatorHoldout: Ignoring 446 ( 1.2%) Users that have less than 1 test interactions
EvaluatorHoldout: Ignoring 145 ( 0.4%) Users that have less than 1 test interactions


In [7]:

# Define RP3beta recommender
rp3_params = {"alpha": 0.336, "beta": 0.146, "topK": 32}  # Example parameters
recommender_rp3 = RP3betaRecommender(URM_train)
recommender_rp3.fit(**rp3_params)

# Define SLIM ElasticNet recommender
slim_params = {"topK": 2305, "l1_ratio": 0.16, "alpha": 0.00069}  # Example parameters
recommender_slim = SLIMElasticNetRecommender(URM_train)
recommender_slim.fit(**slim_params)

# Evaluate individual recommenders
print("RP3beta validation results:")
results_rp3, _ = evaluator_validation.evaluateRecommender(recommender_rp3)
print(results_rp3)

print("SLIM validation results:")
results_slim, _ = evaluator_validation.evaluateRecommender(recommender_slim)
print(results_slim)


RP3betaRecommender: Similarity column 38121 (100.0%), 1978.37 column/sec. Elapsed time 19.27 sec
SLIMElasticNetRecommender: Processed 3766 ( 9.9%) in 5.00 min. Items per second: 12.55
SLIMElasticNetRecommender: Processed 7927 (20.8%) in 10.00 min. Items per second: 13.21
SLIMElasticNetRecommender: Processed 12679 (33.3%) in 15.00 min. Items per second: 14.08
SLIMElasticNetRecommender: Processed 17744 (46.5%) in 20.00 min. Items per second: 14.78
SLIMElasticNetRecommender: Processed 20320 (53.3%) in 25.01 min. Items per second: 13.54
SLIMElasticNetRecommender: Processed 24311 (63.8%) in 30.01 min. Items per second: 13.50
SLIMElasticNetRecommender: Processed 28083 (73.7%) in 35.01 min. Items per second: 13.37
SLIMElasticNetRecommender: Processed 32619 (85.6%) in 40.01 min. Items per second: 13.59
SLIMElasticNetRecommender: Processed 37075 (97.3%) in 45.01 min. Items per second: 13.73
SLIMElasticNetRecommender: Processed 38121 (100.0%) in 46.14 min. Items per second: 13.77
RP3beta validat

In [8]:

# Create a hybrid recommender using weighted similarity matrices
alpha_hybrid = 0.56  # Weight for combining RP3 and SLIM similarities
hybrid_similarity = (1 - alpha_hybrid) * recommender_rp3.W_sparse + alpha_hybrid * recommender_slim.W_sparse

recommender_hybrid = ItemKNNCustomSimilarityRecommender(URM_train)
recommender_hybrid.fit(hybrid_similarity)


In [9]:

# Evaluate the hybrid recommender
print("Hybrid recommender validation results:")
results_hybrid, _ = evaluator_validation.evaluateRecommender(recommender_hybrid)
print(results_hybrid)

Hybrid recommender validation results:
EvaluatorHoldout: Processed 35290 (100.0%) in 37.00 sec. Users per second: 954
       PRECISION PRECISION_RECALL_MIN_DEN    RECALL     MAP MAP_MIN_DEN  \
cutoff                                                                    
10      0.069776                  0.12318  0.111606  0.0326    0.056613   

             MRR      NDCG        F1  HIT_RATE ARHR_ALL_HITS  ...  \
cutoff                                                        ...   
10      0.209684  0.110782  0.085868  0.445735      0.259326  ...   

       COVERAGE_USER COVERAGE_USER_HIT USERS_IN_GT DIVERSITY_GINI  \
cutoff                                                              
10           0.98752          0.440172     0.98752       0.151448   

       SHANNON_ENTROPY RATIO_DIVERSITY_HERFINDAHL RATIO_DIVERSITY_GINI  \
cutoff                                                                   
10           12.757679                    0.99973             0.250209   

       RATIO_SHAN

In [13]:
folder_path = "C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\"
import os
import csv
os.makedirs(folder_path, exist_ok=True)

output_file = os.path.join(folder_path, "sample_submission.csv")
# Generate recommendations for target users

with open(output_file, 'w') as f:
    f.write("user_id,item_list\n")
    for user_id in target_users['user_id']:
        recommended_items = recommender_hybrid.recommend(user_id, cutoff=10)
        f.write(f"{user_id},{' '.join(map(str, recommended_items))}\n")

print(f"Recommendations saved to {output_file}")

Recommendations saved to C:\Users\VOLKAN MAZLUM\Desktop\Proje\sample_submission.csv


In [14]:
import os
import csv
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix


import pandas as pd
import scipy.sparse as sp
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from Evaluation.Evaluator import EvaluatorHoldout
from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender
from Recommenders.KNN.ItemKNNCustomSimilarityRecommender import ItemKNNCustomSimilarityRecommender

# Veri yükleme
data_train = pd.read_csv("C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\data_train.csv")
icm_metadata = pd.read_csv("C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\data_ICM_metadata.csv")
target_users = pd.read_csv("C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\data_target_users_test.csv")

# Sparse matrisi oluşturma
n_users = data_train['user_id'].nunique()
n_items = data_train['item_id'].nunique()

URM_all = csr_matrix((data_train['data'], 
                      (data_train['user_id'], data_train['item_id'])),
                     shape=(n_users, n_items))
n_features = icm_metadata['feature_id'].nunique()

# Create the sparse matrix for ICM (Item-Feature interactions)
ICM_all = csr_matrix((icm_metadata['data'], 
                      (icm_metadata['item_id'], icm_metadata['feature_id'])),
                     shape=(n_items, n_features))

# Split data fonksiyonu
def split_data(URM_all, train_percentage=0.8):
    URM_train_validation, URM_test = split_train_in_two_percentage_global_sample(URM_all, train_percentage)
    URM_train, URM_validation = split_train_in_two_percentage_global_sample(URM_train_validation, train_percentage)
    return URM_train, URM_validation, URM_test

# En iyi alpha'yı bulma fonksiyonu
def find_best_alpha(recommender1, recommender2, evaluator, alpha_range, URM_train):
    best_alpha, best_map = None, 0.0
    for alpha in alpha_range:
        hybrid_similarity = (1 - alpha) * recommender1.W_sparse + alpha * recommender2.W_sparse
        hybrid_recommender = ItemKNNCustomSimilarityRecommender(URM_train)
        hybrid_recommender.fit(hybrid_similarity)
        
        results_df, _ = evaluator.evaluateRecommender(hybrid_recommender)
        map_value = results_df["MAP"].values[0]
        
        if map_value > best_map:
            best_alpha, best_map = alpha, map_value
    
    return best_alpha, best_map

# Hiperparametre arama fonksiyonu
def tune_recommender_params(recommender_class, params_list, evaluator, URM_train):
    best_index, max_map = None, 0.0
    for i, params in enumerate(params_list):
        recommender = recommender_class(URM_train)
        recommender.fit(**params)
        
        results_df, _ = evaluator.evaluateRecommender(recommender)
        map_value = results_df["MAP"].item()
        
        if map_value > max_map:
            best_index, max_map = i, map_value
    
    return best_index

# Ana iş akışı
def main(URM_all, params_rp3, params_slim, alpha_range):
    # Veri bölme
    URM_train, URM_validation, URM_test = split_data(URM_all)
    
    # Değerlendirme objeleri
    evaluator_validation = EvaluatorHoldout(URM_validation, cutoff_list=[10])
    evaluator_test = EvaluatorHoldout(URM_test, cutoff_list=[10])
    
    # RP3 için en iyi parametre seçimi
    rp3_index = tune_recommender_params(RP3betaRecommender, params_rp3, evaluator_validation, URM_train)
    best_rp3_params = params_rp3[rp3_index]
    rp3_recommender = RP3betaRecommender(URM_train)
    rp3_recommender.fit(**best_rp3_params)
    
    # SLIM için en iyi parametre seçimi
    slim_index = tune_recommender_params(SLIMElasticNetRecommender, params_slim, evaluator_validation, URM_train)
    best_slim_params = params_slim[slim_index]
    slim_recommender = SLIMElasticNetRecommender(URM_train)
    slim_recommender.fit(**best_slim_params)
    
    # En iyi alpha değerini bulma
    best_alpha, best_map = find_best_alpha(rp3_recommender, slim_recommender, evaluator_validation, alpha_range,URM_train)
    print(f"Best alpha: {best_alpha}, Best MAP: {best_map}")
    
    # Tüm veriyle yeniden eğitim ve tahmin
    rp3_recommender = RP3betaRecommender(URM_all)
    rp3_recommender.fit(**best_rp3_params)
    slim_recommender = SLIMElasticNetRecommender(URM_all)
    slim_recommender.fit(**best_slim_params)
    
    hybrid_similarity = (1 - best_alpha) * rp3_recommender.W_sparse + best_alpha * slim_recommender.W_sparse
    hybrid_recommender = ItemKNNCustomSimilarityRecommender(URM_all)
    hybrid_recommender.fit(hybrid_similarity)
    
    return hybrid_recommender


In [8]:

# Örnek parametreler
params_rp3 = [{"alpha": 0.33, "beta": 0.14, "topK": 32}, {"alpha": 0.32, "beta": 0.12, "topK": 30}, {"alpha": 0.39, "beta": 0.14, "topK": 28}, {"alpha": 0.32, "beta": 0.14, "topK": 36}, {"alpha": 0.35, "beta": 0.16, "topK": 35}]
params_slim = [{"topK": 2305, "l1_ratio": 0.15, "alpha": 0.0006}, {"topK": 2327, "l1_ratio": 0.16, "alpha": 0.0007}, {"topK": 1827, "l1_ratio": 0.19, "alpha": 0.0005}, {"topK": 2007, "l1_ratio": 0.11, "alpha": 0.0009}, {"topK": 3327, "l1_ratio": 0.17, "alpha": 0.0012}]
alpha_range = np.arange(0.1, 0.68, 0.01)

# Çalıştırma
hybrid_recommender = main(URM_all, params_rp3, params_slim, alpha_range)

# Sonuçları kaydetme
output_folder = "C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\"
os.makedirs(output_folder, exist_ok=True)
output_file = os.path.join(output_folder, "sample_submission11.csv")

with open(output_file, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['user_id', 'item_list'])
    for user_id in target_users['user_id']:
        recommendations = hybrid_recommender.recommend(user_id, cutoff=10)
        writer.writerow([user_id, " ".join(map(str, recommendations))])

print(f"Results saved to {output_file}")


EvaluatorHoldout: Ignoring 2609 ( 7.3%) Users that have less than 1 test interactions
EvaluatorHoldout: Ignoring 1988 ( 5.6%) Users that have less than 1 test interactions
RP3betaRecommender: Similarity column 38121 (100.0%), 1801.20 column/sec. Elapsed time 21.16 sec
EvaluatorHoldout: Processed 33127 (100.0%) in 29.80 sec. Users per second: 1112
RP3betaRecommender: Similarity column 38121 (100.0%), 1820.36 column/sec. Elapsed time 20.94 sec
EvaluatorHoldout: Processed 33127 (100.0%) in 26.90 sec. Users per second: 1231
RP3betaRecommender: Similarity column 38121 (100.0%), 1908.62 column/sec. Elapsed time 19.97 sec
EvaluatorHoldout: Processed 33127 (100.0%) in 26.96 sec. Users per second: 1229
RP3betaRecommender: Similarity column 38121 (100.0%), 1824.40 column/sec. Elapsed time 20.90 sec
EvaluatorHoldout: Processed 33127 (100.0%) in 28.76 sec. Users per second: 1152
RP3betaRecommender: Similarity column 38121 (100.0%), 1908.62 column/sec. Elapsed time 19.97 sec
EvaluatorHoldout: Proce

EvaluatorHoldout: Processed 33127 (100.0%) in 31.46 sec. Users per second: 1053
EvaluatorHoldout: Processed 33127 (100.0%) in 34.88 sec. Users per second: 950
EvaluatorHoldout: Processed 33127 (100.0%) in 33.62 sec. Users per second: 985
EvaluatorHoldout: Processed 33127 (100.0%) in 33.82 sec. Users per second: 980
EvaluatorHoldout: Processed 33127 (100.0%) in 31.52 sec. Users per second: 1051
EvaluatorHoldout: Processed 33127 (100.0%) in 32.22 sec. Users per second: 1028
EvaluatorHoldout: Processed 33127 (100.0%) in 34.57 sec. Users per second: 958
EvaluatorHoldout: Processed 33127 (100.0%) in 32.39 sec. Users per second: 1023
EvaluatorHoldout: Processed 33127 (100.0%) in 32.25 sec. Users per second: 1027
EvaluatorHoldout: Processed 33127 (100.0%) in 33.75 sec. Users per second: 981
EvaluatorHoldout: Processed 33127 (100.0%) in 33.61 sec. Users per second: 986
EvaluatorHoldout: Processed 33127 (100.0%) in 33.17 sec. Users per second: 999
EvaluatorHoldout: Processed 33127 (100.0%) in 3

In [10]:

# Örnek parametreler
params_rp3 = [{"alpha": 0.38, "beta": 0.25, "topK": 87},{"alpha": 0.32, "beta": 0.11, "topK": 30},{"alpha": 0.35, "beta": 0.24, "topK": 28},{"alpha": 0.54, "beta": 0.11, "topK": 49},{"alpha": 0.35, "beta": 0.21, "topK": 45},{"alpha": 0.33, "beta": 0.14, "topK": 32}, {"alpha": 0.32, "beta": 0.12, "topK": 30}, {"alpha": 0.39, "beta": 0.14, "topK": 28}, {"alpha": 0.32, "beta": 0.14, "topK": 36}, {"alpha": 0.35, "beta": 0.16, "topK": 35}, {"alpha": 0.55, "beta": 0.10, "topK": 26}]
params_slim = [{"topK": 2305, "l1_ratio": 0.15, "alpha": 0.0006}, {"topK": 2327, "l1_ratio": 0.16, "alpha": 0.0007}, {"topK": 1827, "l1_ratio": 0.19, "alpha": 0.0005}, {"topK": 2007, "l1_ratio": 0.11, "alpha": 0.0009}, {"topK": 3327, "l1_ratio": 0.17, "alpha": 0.0012}]
alpha_range = np.arange(0.1, 0.68, 0.01)

# Çalıştırma
hybrid_recommender = main(URM_all, params_rp3, params_slim, alpha_range)

# Sonuçları kaydetme
output_folder = "C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\"
os.makedirs(output_folder, exist_ok=True)
output_file = os.path.join(output_folder, "sample_submission12.csv")

with open(output_file, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['user_id', 'item_list'])
    for user_id in target_users['user_id']:
        recommendations = hybrid_recommender.recommend(user_id, cutoff=10)
        writer.writerow([user_id, " ".join(map(str, recommendations))])

print(f"Results saved to {output_file}")


EvaluatorHoldout: Ignoring 2573 ( 7.2%) Users that have less than 1 test interactions
EvaluatorHoldout: Ignoring 1888 ( 5.3%) Users that have less than 1 test interactions
RP3betaRecommender: Similarity column 38121 (100.0%), 1086.14 column/sec. Elapsed time 35.10 sec
EvaluatorHoldout: Processed 33163 (100.0%) in 35.54 sec. Users per second: 933
RP3betaRecommender: Similarity column 38121 (100.0%), 1536.62 column/sec. Elapsed time 24.81 sec
EvaluatorHoldout: Processed 33163 (100.0%) in 33.48 sec. Users per second: 991
RP3betaRecommender: Similarity column 38121 (100.0%), 1628.38 column/sec. Elapsed time 23.41 sec
EvaluatorHoldout: Processed 33163 (100.0%) in 33.75 sec. Users per second: 982
RP3betaRecommender: Similarity column 38121 (100.0%), 1722.45 column/sec. Elapsed time 22.13 sec
EvaluatorHoldout: Processed 33163 (100.0%) in 28.67 sec. Users per second: 1157
RP3betaRecommender: Similarity column 38121 (100.0%), 1830.25 column/sec. Elapsed time 20.83 sec
EvaluatorHoldout: Processe

SLIMElasticNetRecommender: Processed 18358 (48.2%) in 30.00 min. Items per second: 10.20
SLIMElasticNetRecommender: Processed 21793 (57.2%) in 35.00 min. Items per second: 10.38
SLIMElasticNetRecommender: Processed 25134 (65.9%) in 40.00 min. Items per second: 10.47
SLIMElasticNetRecommender: Processed 28427 (74.6%) in 45.00 min. Items per second: 10.53
SLIMElasticNetRecommender: Processed 31834 (83.5%) in 50.01 min. Items per second: 10.61
SLIMElasticNetRecommender: Processed 35354 (92.7%) in 55.01 min. Items per second: 10.71
SLIMElasticNetRecommender: Processed 38121 (100.0%) in 59.26 min. Items per second: 10.72
EvaluatorHoldout: Processed 33163 (100.0%) in 32.21 sec. Users per second: 1030
EvaluatorHoldout: Processed 33163 (100.0%) in 34.04 sec. Users per second: 974
EvaluatorHoldout: Processed 33163 (100.0%) in 35.62 sec. Users per second: 931
EvaluatorHoldout: Processed 33163 (100.0%) in 31.91 sec. Users per second: 1039
EvaluatorHoldout: Processed 33163 (100.0%) in 34.38 sec. U

In [11]:

# Örnek parametreler
params_rp3 = [{"alpha": 0.38, "beta": 0.25, "topK": 87},{"alpha": 0.32, "beta": 0.11, "topK": 30},{"alpha": 0.35, "beta": 0.24, "topK": 28},{"alpha": 0.54, "beta": 0.11, "topK": 49},{"alpha": 0.35, "beta": 0.21, "topK": 45},{"alpha": 0.33, "beta": 0.14, "topK": 32}, {"alpha": 0.32, "beta": 0.12, "topK": 30}, {"alpha": 0.39, "beta": 0.14, "topK": 28}, {"alpha": 0.32, "beta": 0.14, "topK": 36}, {"alpha": 0.35, "beta": 0.16, "topK": 35}, {"alpha": 0.55, "beta": 0.10, "topK": 26}]
params_slim = [{"topK": 2455, "l1_ratio": 0.20, "alpha": 0.0010},{"topK": 2305, "l1_ratio": 0.19, "alpha": 0.0018},{"topK": 2305, "l1_ratio": 0.15, "alpha": 0.0006}, {"topK": 2327, "l1_ratio": 0.16, "alpha": 0.0007}, {"topK": 1827, "l1_ratio": 0.19, "alpha": 0.0005}, {"topK": 2007, "l1_ratio": 0.11, "alpha": 0.0009}, {"topK": 3327, "l1_ratio": 0.17, "alpha": 0.0012}]
alpha_range = np.arange(0.1, 0.68, 0.01)

# Çalıştırma
hybrid_recommender = main(URM_all, params_rp3, params_slim, alpha_range)

# Sonuçları kaydetme
output_folder = "C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\"
os.makedirs(output_folder, exist_ok=True)
output_file = os.path.join(output_folder, "sample_submission10.csv")

with open(output_file, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['user_id', 'item_list'])
    for user_id in target_users['user_id']:
        recommendations = hybrid_recommender.recommend(user_id, cutoff=10)
        writer.writerow([user_id, " ".join(map(str, recommendations))])

print(f"Results saved to {output_file}")


EvaluatorHoldout: Ignoring 2583 ( 7.2%) Users that have less than 1 test interactions
EvaluatorHoldout: Ignoring 1933 ( 5.4%) Users that have less than 1 test interactions
RP3betaRecommender: Similarity column 38121 (100.0%), 1528.12 column/sec. Elapsed time 24.95 sec
EvaluatorHoldout: Processed 33153 (100.0%) in 34.13 sec. Users per second: 972
RP3betaRecommender: Similarity column 38121 (100.0%), 1731.16 column/sec. Elapsed time 22.02 sec
EvaluatorHoldout: Processed 33153 (100.0%) in 27.80 sec. Users per second: 1193
RP3betaRecommender: Similarity column 38121 (100.0%), 1811.00 column/sec. Elapsed time 21.05 sec
EvaluatorHoldout: Processed 33153 (100.0%) in 31.68 sec. Users per second: 1046
RP3betaRecommender: Similarity column 38121 (100.0%), 1848.33 column/sec. Elapsed time 20.62 sec
EvaluatorHoldout: Processed 33153 (100.0%) in 31.87 sec. Users per second: 1040
RP3betaRecommender: Similarity column 38121 (100.0%), 1738.54 column/sec. Elapsed time 21.93 sec
EvaluatorHoldout: Proces

SLIMElasticNetRecommender: Processed 33236 (87.2%) in 55.01 min. Items per second: 10.07
SLIMElasticNetRecommender: Processed 36462 (95.6%) in 1.00 hour. Items per second: 10.13
SLIMElasticNetRecommender: Processed 38121 (100.0%) in 1.05 hour. Items per second: 10.12
EvaluatorHoldout: Processed 33153 (100.0%) in 35.58 sec. Users per second: 932
SLIMElasticNetRecommender: Processed 4147 (10.9%) in 5.00 min. Items per second: 13.82
SLIMElasticNetRecommender: Processed 8493 (22.3%) in 10.00 min. Items per second: 14.15
SLIMElasticNetRecommender: Processed 12993 (34.1%) in 15.00 min. Items per second: 14.43
SLIMElasticNetRecommender: Processed 17612 (46.2%) in 20.00 min. Items per second: 14.67
SLIMElasticNetRecommender: Processed 22318 (58.5%) in 25.00 min. Items per second: 14.88
SLIMElasticNetRecommender: Processed 26860 (70.5%) in 30.00 min. Items per second: 14.92
SLIMElasticNetRecommender: Processed 31514 (82.7%) in 35.00 min. Items per second: 15.00
SLIMElasticNetRecommender: Proces

SLIMElasticNetRecommender: Processed 26439 (69.4%) in 1.00 hour. Items per second: 7.34
SLIMElasticNetRecommender: Processed 28766 (75.5%) in 1.08 hour. Items per second: 7.37
SLIMElasticNetRecommender: Processed 31077 (81.5%) in 1.17 hour. Items per second: 7.40
SLIMElasticNetRecommender: Processed 33554 (88.0%) in 1.25 hour. Items per second: 7.45
SLIMElasticNetRecommender: Processed 35924 (94.2%) in 1.33 hour. Items per second: 7.48
SLIMElasticNetRecommender: Processed 38121 (100.0%) in 1.41 hour. Items per second: 7.48
Results saved to C:\Users\VOLKAN MAZLUM\Desktop\Proje\sample_submission10.csv


In [13]:

# Örnek parametreler
params_rp3 = [{"alpha": 0.3365280890390189, "beta": 0.14559541984971103, "topK": 32},
    {"alpha": 0.33543320721534375, "beta": 0.13643832833854405, "topK": 31},
    {"alpha": 0.3227577390094315, "beta": 0.12631978308173344, "topK": 32},
    {"alpha": 0.2617995480887691, "beta": 0.22418804987168886, "topK": 30},
    {'alpha': 0.28585975670634217, 'beta': 0.13388489746818844, 'topK': 33},
    {'alpha': 0.3361086178381283, 'beta': 0.13949133462799973, 'topK': 29}, 
    {"alpha": 0.32, "beta": 0.12, "topK": 30}, 
    {"alpha": 0.39, "beta": 0.14, "topK": 28}, 
    {"alpha": 0.32, "beta": 0.14, "topK": 36}, 
    {"alpha": 0.35, "beta": 0.16, "topK": 35}, 
    {"alpha": 0.55, "beta": 0.10, "topK": 26}]
params_slim = [{'topK': 2305, 'l1_ratio': 0.15984659917724292, 'alpha': 0.0006895792558081994},
    {'topK': 2339, 'l1_ratio': 0.15486907556362542, 'alpha': 0.0006851706335261893},
    {"topK": 2327, "l1_ratio": 0.15346747937279875, "alpha": 0.000677913689441996},
    {'topK': 2427, 'l1_ratio': 0.14931044947790595, 'alpha': 0.0007442377587336158},
    {'topK': 2310, 'l1_ratio': 0.1519150334556062, 'alpha': 0.0006862030334431442}
    ]
alpha_range = np.arange(0.25, 0.70, 0.01)

# Çalıştırma
hybrid_recommender = main(URM_all, params_rp3, params_slim, alpha_range)

# Sonuçları kaydetme
output_folder = "C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\"
os.makedirs(output_folder, exist_ok=True)
output_file = os.path.join(output_folder, "sample_submission14.csv")

with open(output_file, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['user_id', 'item_list'])
    for user_id in target_users['user_id']:
        recommendations = hybrid_recommender.recommend(user_id, cutoff=10)
        writer.writerow([user_id, " ".join(map(str, recommendations))])

print(f"Results saved to {output_file}")


EvaluatorHoldout: Ignoring 2529 ( 7.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Ignoring 1923 ( 5.4%) Users that have less than 1 test interactions
RP3betaRecommender: Similarity column 38121 (100.0%), 941.11 column/sec. Elapsed time 40.51 sec
EvaluatorHoldout: Processed 33207 (100.0%) in 41.24 sec. Users per second: 805
RP3betaRecommender: Similarity column 38121 (100.0%), 1469.32 column/sec. Elapsed time 25.94 sec
EvaluatorHoldout: Processed 33207 (100.0%) in 37.23 sec. Users per second: 892
RP3betaRecommender: Similarity column 38121 (100.0%), 1912.47 column/sec. Elapsed time 19.93 sec
EvaluatorHoldout: Processed 33207 (100.0%) in 30.76 sec. Users per second: 1080
RP3betaRecommender: Similarity column 38121 (100.0%), 2000.26 column/sec. Elapsed time 19.06 sec
EvaluatorHoldout: Processed 33207 (100.0%) in 29.76 sec. Users per second: 1116
RP3betaRecommender: Similarity column 38121 (100.0%), 1968.00 column/sec. Elapsed time 19.37 sec
EvaluatorHoldout: Processe

SLIMElasticNetRecommender: Processed 38121 (100.0%) in 1.08 hour. Items per second: 9.82
EvaluatorHoldout: Processed 33207 (100.0%) in 33.70 sec. Users per second: 985
SLIMElasticNetRecommender: Processed 2477 ( 6.5%) in 5.00 min. Items per second: 8.25
SLIMElasticNetRecommender: Processed 5364 (14.1%) in 10.00 min. Items per second: 8.94
SLIMElasticNetRecommender: Processed 8162 (21.4%) in 15.00 min. Items per second: 9.07
SLIMElasticNetRecommender: Processed 10997 (28.8%) in 20.01 min. Items per second: 9.16
SLIMElasticNetRecommender: Processed 13980 (36.7%) in 25.01 min. Items per second: 9.32
SLIMElasticNetRecommender: Processed 17189 (45.1%) in 30.01 min. Items per second: 9.55
SLIMElasticNetRecommender: Processed 20216 (53.0%) in 35.01 min. Items per second: 9.62
SLIMElasticNetRecommender: Processed 23222 (60.9%) in 40.01 min. Items per second: 9.67
SLIMElasticNetRecommender: Processed 26326 (69.1%) in 45.01 min. Items per second: 9.75
SLIMElasticNetRecommender: Processed 29257 (

In [16]:

# Örnek parametreler
params_rp3 = [{"alpha": 0.3365280890390189, "beta": 0.14559541984971103, "topK": 32},
    {"alpha": 0.33543320721534375, "beta": 0.13643832833854405, "topK": 31},
    {"alpha": 0.3227577390094315, "beta": 0.12631978308173344, "topK": 32},
    {"alpha": 0.2617995480887691, "beta": 0.22418804987168886, "topK": 30},
    {'alpha': 0.28585975670634217, 'beta': 0.13388489746818844, 'topK': 33},
    {'alpha': 0.3361086178381283, 'beta': 0.13949133462799973, 'topK': 29}
]
params_slim = [{'topK': 2305, 'l1_ratio': 0.15984659917724292, 'alpha': 0.0006895792558081994},
    {'topK': 2339, 'l1_ratio': 0.15486907556362542, 'alpha': 0.0006851706335261893},
    {"topK": 2327, "l1_ratio": 0.15346747937279875, "alpha": 0.000677913689441996}
    ]
alpha_range = np.arange(0.25, 0.70, 0.01)


# Çalıştırma
hybrid_recommender = main(URM_all, params_rp3, params_slim, alpha_range)

# Sonuçları kaydetme
output_folder = "C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\"
os.makedirs(output_folder, exist_ok=True)
output_file = os.path.join(output_folder, "sample_submission18.csv")

with open(output_file, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['user_id', 'item_list'])
    for user_id in target_users['user_id']:
        recommendations = hybrid_recommender.recommend(user_id, cutoff=10)
        writer.writerow([user_id, " ".join(map(str, recommendations))])

print(f"Results saved to {output_file}")


EvaluatorHoldout: Ignoring 436 ( 1.2%) Users that have less than 1 test interactions
EvaluatorHoldout: Ignoring 152 ( 0.4%) Users that have less than 1 test interactions
RP3betaRecommender: Similarity column 38121 (100.0%), 2116.68 column/sec. Elapsed time 18.01 sec
EvaluatorHoldout: Processed 35300 (100.0%) in 32.18 sec. Users per second: 1097
RP3betaRecommender: Similarity column 38121 (100.0%), 2055.55 column/sec. Elapsed time 18.55 sec
EvaluatorHoldout: Processed 35300 (100.0%) in 31.85 sec. Users per second: 1108
RP3betaRecommender: Similarity column 38121 (100.0%), 1963.17 column/sec. Elapsed time 19.42 sec
EvaluatorHoldout: Processed 35300 (100.0%) in 32.78 sec. Users per second: 1077
RP3betaRecommender: Similarity column 38121 (100.0%), 2078.80 column/sec. Elapsed time 18.34 sec
EvaluatorHoldout: Processed 35300 (100.0%) in 32.62 sec. Users per second: 1082
RP3betaRecommender: Similarity column 38121 (100.0%), 2023.38 column/sec. Elapsed time 18.84 sec
EvaluatorHoldout: Process

EvaluatorHoldout: Processed 35300 (100.0%) in 35.54 sec. Users per second: 993
EvaluatorHoldout: Processed 35300 (100.0%) in 34.37 sec. Users per second: 1027
EvaluatorHoldout: Processed 35300 (100.0%) in 33.82 sec. Users per second: 1044
Best alpha: 0.42000000000000015, Best MAP: 0.03225080489230539
RP3betaRecommender: Similarity column 38121 (100.0%), 1704.40 column/sec. Elapsed time 22.37 sec
SLIMElasticNetRecommender: Processed 2014 ( 5.3%) in 5.00 min. Items per second: 6.71
SLIMElasticNetRecommender: Processed 4067 (10.7%) in 10.00 min. Items per second: 6.78
SLIMElasticNetRecommender: Processed 6251 (16.4%) in 15.00 min. Items per second: 6.94
SLIMElasticNetRecommender: Processed 8499 (22.3%) in 20.01 min. Items per second: 7.08
SLIMElasticNetRecommender: Processed 10753 (28.2%) in 25.01 min. Items per second: 7.17
SLIMElasticNetRecommender: Processed 13098 (34.4%) in 30.01 min. Items per second: 7.27
SLIMElasticNetRecommender: Processed 15509 (40.7%) in 35.01 min. Items per sec

In [4]:
import os
import csv
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from Evaluation.Evaluator import EvaluatorHoldout
from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender
from Recommenders.KNN.ItemKNNCustomSimilarityRecommender import ItemKNNCustomSimilarityRecommender

# Load data
data_train = pd.read_csv("C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\data_train.csv")
icm_metadata = pd.read_csv("C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\data_ICM_metadata.csv")
target_users = pd.read_csv("C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\data_target_users_test.csv")

# Create sparse matrices
n_users = data_train['user_id'].nunique()
n_items = data_train['item_id'].nunique()

URM_all = csr_matrix((data_train['data'], 
                      (data_train['user_id'], data_train['item_id'])),
                     shape=(n_users, n_items))

n_features = icm_metadata['feature_id'].nunique()

ICM_all = csr_matrix((icm_metadata['data'], 
                      (icm_metadata['item_id'], icm_metadata['feature_id'])),
                     shape=(n_items, n_features))

# Split data function
def split_data(URM_all, train_percentage=0.9):
    URM_train_validation, URM_test = split_train_in_two_percentage_global_sample(URM_all, train_percentage)
    URM_train, URM_validation = split_train_in_two_percentage_global_sample(URM_train_validation, train_percentage)
    return URM_train, URM_validation, URM_test

# Find best alpha function
def find_best_alpha(recommender1, recommender2, evaluator, alpha_range, URM_train, ICM_weight):
    best_alpha, best_map = None, 0.0
    for alpha in alpha_range:
        hybrid_similarity = ((1 - alpha) * recommender1.W_sparse + 
                             alpha * recommender2.W_sparse +
                             ICM_weight * ICM_all.T @ ICM_all)
                             
        hybrid_recommender = ItemKNNCustomSimilarityRecommender(URM_train)
        hybrid_recommender.fit(hybrid_similarity)
        
        results_df, _ = evaluator.evaluateRecommender(hybrid_recommender)
        map_value = results_df["MAP"].values[0]
        
        if map_value > best_map:
            best_alpha, best_map = alpha, map_value
    
    return best_alpha, best_map

# Hyperparameter tuning function
def tune_recommender_params(recommender_class, params_list, evaluator, URM_train):
    best_index, max_map = None, 0.0
    for i, params in enumerate(params_list):
        recommender = recommender_class(URM_train)
        recommender.fit(**params)
        
        results_df, _ = evaluator.evaluateRecommender(recommender)
        map_value = results_df["MAP"].item()
        
        if map_value > max_map:
            best_index, max_map = i, map_value
    
    return best_index

# Main workflow
def main(URM_all, params_rp3, params_slim, alpha_range, ICM_weight):
    # Split data
    URM_train, URM_validation, URM_test = split_data(URM_all)
    
    # Evaluation objects
    evaluator_validation = EvaluatorHoldout(URM_validation, cutoff_list=[10])
    evaluator_test = EvaluatorHoldout(URM_test, cutoff_list=[10])
    
    # RP3 best parameters
    rp3_index = tune_recommender_params(RP3betaRecommender, params_rp3, evaluator_validation, URM_train)
    best_rp3_params = params_rp3[rp3_index]
    rp3_recommender = RP3betaRecommender(URM_train)
    rp3_recommender.fit(**best_rp3_params)
    
    # SLIM best parameters
    slim_index = tune_recommender_params(SLIMElasticNetRecommender, params_slim, evaluator_validation, URM_train)
    best_slim_params = params_slim[slim_index]
    slim_recommender = SLIMElasticNetRecommender(URM_train)
    slim_recommender.fit(**best_slim_params)
    
    # Find best alpha
    best_alpha, best_map = find_best_alpha(rp3_recommender, slim_recommender, evaluator_validation, alpha_range, URM_train, ICM_weight)
    print(f"Best alpha: {best_alpha}, Best MAP: {best_map}")
    
    # Re-train on full data
    rp3_recommender = RP3betaRecommender(URM_all)
    rp3_recommender.fit(**best_rp3_params)
    slim_recommender = SLIMElasticNetRecommender(URM_all)
    slim_recommender.fit(**best_slim_params)
    
    hybrid_similarity = ((1 - best_alpha) * rp3_recommender.W_sparse + 
                         best_alpha * slim_recommender.W_sparse +
                         ICM_weight * ICM_all.T @ ICM_all)
                         
    hybrid_recommender = ItemKNNCustomSimilarityRecommender(URM_all)
    hybrid_recommender.fit(hybrid_similarity)
    
    return hybrid_recommender 


In [5]:

# Example parameters
params_rp3 = [{"alpha": 0.33, "beta": 0.14, "topK": 32}, {"alpha": 0.32, "beta": 0.12, "topK": 30}, {"alpha": 0.39, "beta": 0.14, "topK": 28}, {"alpha": 0.32, "beta": 0.14, "topK": 36}, {"alpha": 0.35, "beta": 0.16, "topK": 35}]
params_slim = [{"topK": 2305, "l1_ratio": 0.15, "alpha": 0.0006}, {"topK": 2327, "l1_ratio": 0.16, "alpha": 0.0007}, {"topK": 1827, "l1_ratio": 0.19, "alpha": 0.0005}, {"topK": 2007, "l1_ratio": 0.11, "alpha": 0.0009}, {"topK": 3327, "l1_ratio": 0.17, "alpha": 0.0012}]
alpha_range = np.arange(0.1, 0.68, 0.01)
ICM_weight = 0.2

# Run
hybrid_recommender = main(URM_all, params_rp3, params_slim, alpha_range, ICM_weight)


EvaluatorHoldout: Ignoring 2561 ( 7.2%) Users that have less than 1 test interactions
EvaluatorHoldout: Ignoring 1926 ( 5.4%) Users that have less than 1 test interactions
RP3betaRecommender: Similarity column 38121 (100.0%), 831.65 column/sec. Elapsed time 45.84 sec


KeyboardInterrupt: 

In [6]:
!pip install optuna

   ---------------------------------------- 364.4/364.4 kB 1.7 MB/s eta 0:00:00
   ---------------------------------------- 233.5/233.5 kB 1.8 MB/s eta 0:00:00
   ---------------------------------------- 78.6/78.6 kB 1.1 MB/s eta 0:00:00


DEPRECATION: pyodbc 4.0.0-unsupported has a non-standard version number. pip 24.0 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pyodbc or contact the author to suggest that they release a version with a conforming version number. Discussion can be found at https://github.com/pypa/pip/issues/12063

[notice] A new release of pip is available: 23.3.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:

# Save results
output_folder = "C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\"
os.makedirs(output_folder, exist_ok=True)
output_file = os.path.join(output_folder, "sample_submission_icm.csv")

with open(output_file, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['user_id', 'item_list'])
    for user_id in target_users['user_id']:
        recommendations = hybrid_recommender.recommend(user_id, cutoff=10)
        writer.writerow([user_id, " ".join(map(str, recommendations))])

print(f"Results saved to {output_file}")


In [14]:
import os
import csv
import numpy as np
import pandas as pd
import optuna
from scipy.sparse import csr_matrix

from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from Evaluation.Evaluator import EvaluatorHoldout
from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender
from Recommenders.KNN.ItemKNNCustomSimilarityRecommender import ItemKNNCustomSimilarityRecommender

# Load data
data_train = pd.read_csv("C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\data_train.csv")
icm_metadata = pd.read_csv("C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\data_ICM_metadata.csv")
target_users = pd.read_csv("C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\data_target_users_test.csv")

# Create sparse matrices
n_users = data_train['user_id'].nunique()
n_items = data_train['item_id'].nunique()

URM_all = csr_matrix((data_train['data'], 
                      (data_train['user_id'], data_train['item_id'])),
                     shape=(n_users, n_items))

n_features = icm_metadata['feature_id'].nunique()

ICM_all = csr_matrix((icm_metadata['data'], 
                      (icm_metadata['item_id'], icm_metadata['feature_id'])),
                     shape=(n_items, n_features))

# Split data function
def split_data(URM_all, train_percentage=0.8):
    URM_train_validation, URM_test = split_train_in_two_percentage_global_sample(URM_all, train_percentage)
    URM_train, URM_validation = split_train_in_two_percentage_global_sample(URM_train_validation, train_percentage)
    return URM_train, URM_validation, URM_test

# Optuna objective function
def objective(trial):
    # RP3beta hyperparameters
    alpha_rp3 = trial.suggest_float("alpha_rp3", 0.1, 0.9)
    beta_rp3 = trial.suggest_float("beta_rp3", 0.1, 0.9)
    topK_rp3 = trial.suggest_int("topK_rp3", 10, 1000)

    # SLIMElasticNet hyperparameters
    alpha_slim = trial.suggest_float("alpha_slim", 1e-5, 1e-2, log=True)
    l1_ratio_slim = trial.suggest_float("l1_ratio_slim", 0.01, 0.3)
    topK_slim = trial.suggest_int("topK_slim", 10, 1000)

    # Hybrid alpha and ICM weight
    hybrid_alpha = trial.suggest_float("hybrid_alpha", 0.1, 0.9)
    ICM_weight = trial.suggest_float("ICM_weight", 0.1, 0.5)

    # RP3beta Recommender
    rp3_recommender = RP3betaRecommender(URM_train)
    rp3_recommender.fit(alpha=alpha_rp3, beta=beta_rp3, topK=topK_rp3)

    # SLIMElasticNet Recommender
    slim_recommender = SLIMElasticNetRecommender(URM_train)
    slim_recommender.fit(alpha=alpha_slim, l1_ratio=l1_ratio_slim, topK=topK_slim)

    # Hybrid Recommender
    hybrid_similarity = (
        (1 - hybrid_alpha) * rp3_recommender.W_sparse
        + hybrid_alpha * slim_recommender.W_sparse
        + ICM_weight * ICM_all.T @ ICM_all
    )

    hybrid_recommender = ItemKNNCustomSimilarityRecommender(URM_train)
    hybrid_recommender.fit(hybrid_similarity)

    # Evaluate the recommender
    result_dict, _ = evaluator_validation.evaluateRecommender(hybrid_recommender)
    map_score = result_dict["MAP"].values[0]

    return map_score


# Main workflow
# Main workflow
def main(URM_all, n_trials):
    global URM_train, URM_validation, evaluator_validation
    
    # Split data
    URM_train, URM_validation, URM_test = split_data(URM_all)
    
    # Evaluation objects
    evaluator_validation = EvaluatorHoldout(URM_validation, cutoff_list=[10])
    
    # Optuna study
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)
    
    print("Best hyperparameters:", study.best_params)
    print("Best MAP:", study.best_value)
    
    # Re-train on full data with best parameters
    best_params = study.best_params
    
    rp3_recommender = RP3betaRecommender(URM_all)
    rp3_recommender.fit(alpha=best_params["alpha_rp3"], beta=best_params["beta_rp3"], topK=best_params["topK_rp3"])
    
    slim_recommender = SLIMElasticNetRecommender(URM_all)
    slim_recommender.fit(alpha=best_params["alpha_slim"], l1_ratio=best_params["l1_ratio_slim"], topK=best_params["topK_slim"])
    
    hybrid_similarity = ((1 - best_params["hybrid_alpha"]) * rp3_recommender.W_sparse + 
                         best_params["hybrid_alpha"] * slim_recommender.W_sparse +
                         best_params["ICM_weight"] * ICM_all.T @ ICM_all)
                         
    hybrid_recommender = ItemKNNCustomSimilarityRecommender(URM_all)
    hybrid_recommender.fit(hybrid_similarity)
    
    return hybrid_recommender, study




In [15]:

# Run optimization and final recommendation generation
n_trials = 10  # Number of trials for Optuna
hybrid_recommender, study = main(URM_all, n_trials)


[I 2024-12-10 13:26:03,195] A new study created in memory with name: no-name-80e7e760-e75f-475a-afd5-0d52d4021a2a


EvaluatorHoldout: Ignoring 459 ( 1.3%) Users that have less than 1 test interactions
RP3betaRecommender: Similarity column 38121 (100.0%), 916.79 column/sec. Elapsed time 41.58 sec
SLIMElasticNetRecommender: Processed 7662 (20.1%) in 5.00 min. Items per second: 25.54
SLIMElasticNetRecommender: Processed 15859 (41.6%) in 10.00 min. Items per second: 26.43
SLIMElasticNetRecommender: Processed 23544 (61.8%) in 15.00 min. Items per second: 26.16
SLIMElasticNetRecommender: Processed 30530 (80.1%) in 20.00 min. Items per second: 25.44
SLIMElasticNetRecommender: Processed 38056 (99.8%) in 25.00 min. Items per second: 25.37
SLIMElasticNetRecommender: Processed 38121 (100.0%) in 25.06 min. Items per second: 25.36


[W 2024-12-10 13:52:14,870] Trial 0 failed with parameters: {'alpha_rp3': 0.15353827419265276, 'beta_rp3': 0.5924064629113924, 'topK_rp3': 369, 'alpha_slim': 0.0013866889865389242, 'l1_ratio_slim': 0.22549149664046195, 'topK_slim': 29, 'hybrid_alpha': 0.36992317687391796, 'ICM_weight': 0.20777125367732002} because of the following error: ValueError('inconsistent shapes').
Traceback (most recent call last):
  File "C:\Users\VOLKAN MAZLUM\anaconda3\lib\site-packages\optuna\study\_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\VOLKAN~1\AppData\Local\Temp/ipykernel_12032/1924013524.py", line 65, in objective
    (1 - hybrid_alpha) * rp3_recommender.W_sparse
  File "C:\Users\VOLKAN MAZLUM\anaconda3\lib\site-packages\scipy\sparse\base.py", line 414, in __add__
    raise ValueError("inconsistent shapes")
ValueError: inconsistent shapes
[W 2024-12-10 13:52:14,870] Trial 0 failed with value None.


ValueError: inconsistent shapes

NameError: name 'rp3_recommender' is not defined

In [ ]:

# Save results
output_folder = "C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\"
os.makedirs(output_folder, exist_ok=True)
output_file = os.path.join(output_folder, "sample_submission_optuna.csv")

with open(output_file, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['user_id', 'item_list'])
    for user_id in target_users['user_id']:
        recommendations = hybrid_recommender.recommend(user_id, cutoff=10)
        writer.writerow([user_id, " ".join(map(str, recommendations))])

print(f"Results saved to {output_file}")

In [18]:
import os
import csv
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
import optuna
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from Evaluation.Evaluator import EvaluatorHoldout
from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender
from Recommenders.KNN.ItemKNNCustomSimilarityRecommender import ItemKNNCustomSimilarityRecommender

# Load data
data_train = pd.read_csv("C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\data_train.csv")
target_users = pd.read_csv("C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\data_target_users_test.csv")

# Create sparse matrix
n_users = data_train['user_id'].nunique()
n_items = data_train['item_id'].nunique()

URM_all = csr_matrix((data_train['data'], 
                      (data_train['user_id'], data_train['item_id'])),
                     shape=(n_users, n_items))

# Split data
def split_data(URM_all, train_percentage=0.8):
    URM_train_validation, URM_test = split_train_in_two_percentage_global_sample(URM_all, train_percentage)
    URM_train, URM_validation = split_train_in_two_percentage_global_sample(URM_train_validation, train_percentage)
    return URM_train, URM_validation, URM_test

# Objective function for Optuna
def objective(trial):
    # Suggest hyperparameters for RP3beta
    alpha_rp3 = trial.suggest_float("alpha_rp3", 0.1, 0.9)
    beta_rp3 = trial.suggest_float("beta_rp3", 0.1, 0.9)
    topK_rp3 = trial.suggest_int("topK_rp3", 10, 1000)

    # Suggest hyperparameters for SLIMElasticNet
    alpha_slim = trial.suggest_float("alpha_slim", 1e-5, 1e-2, log=True)
    l1_ratio_slim = trial.suggest_float("l1_ratio_slim", 0.01, 0.3)
    topK_slim = trial.suggest_int("topK_slim", 10, 1000)

    # Suggest weight for hybrid alpha
    hybrid_alpha = trial.suggest_float("hybrid_alpha", 0.1, 0.9)

    # Train RP3beta
    rp3_recommender = RP3betaRecommender(URM_train)
    rp3_recommender.fit(alpha=alpha_rp3, beta=beta_rp3, topK=topK_rp3)

    # Train SLIMElasticNet
    slim_recommender = SLIMElasticNetRecommender(URM_train)
    slim_recommender.fit(alpha=alpha_slim, l1_ratio=l1_ratio_slim, topK=topK_slim)

    # Combine recommendations using hybrid alpha
    hybrid_similarity = ((1 - hybrid_alpha) * rp3_recommender.W_sparse 
                         + hybrid_alpha * slim_recommender.W_sparse)

    hybrid_recommender = ItemKNNCustomSimilarityRecommender(URM_train)
    hybrid_recommender.fit(hybrid_similarity)

    # Evaluate on validation set
    result_dict, _ = evaluator_validation.evaluateRecommender(hybrid_recommender)
    map_score = result_dict["MAP"].values[0]

    return map_score

# Main workflow
def main(URM_all, n_trials):
    global URM_train, URM_validation, evaluator_validation
    
    # Split data
    URM_train, URM_validation, URM_test = split_data(URM_all)
    
    # Evaluation object
    evaluator_validation = EvaluatorHoldout(URM_validation, cutoff_list=[10])
    
    # Optimize hyperparameters with Optuna
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)
    
    print("Best hyperparameters:", study.best_params)
    print("Best MAP:", study.best_value)
    
    # Retrain on full data with best parameters
    best_params = study.best_params
    
    rp3_recommender = RP3betaRecommender(URM_all)
    rp3_recommender.fit(alpha=best_params["alpha_rp3"], beta=best_params["beta_rp3"], topK=best_params["topK_rp3"])
    
    slim_recommender = SLIMElasticNetRecommender(URM_all)
    slim_recommender.fit(alpha=best_params["alpha_slim"], l1_ratio=best_params["l1_ratio_slim"], topK=best_params["topK_slim"])
    
    hybrid_similarity = ((1 - best_params["hybrid_alpha"]) * rp3_recommender.W_sparse 
                         + best_params["hybrid_alpha"] * slim_recommender.W_sparse)
                         
    hybrid_recommender = ItemKNNCustomSimilarityRecommender(URM_all)
    hybrid_recommender.fit(hybrid_similarity)
    
    return hybrid_recommender, study


In [19]:

# Run the pipeline
n_trials = 10  # Number of Optuna trials
hybrid_recommender, study = main(URM_all, n_trials)



[I 2024-12-10 13:56:45,427] A new study created in memory with name: no-name-735d7a3c-695c-41e1-8683-667970cedb78


EvaluatorHoldout: Ignoring 448 ( 1.3%) Users that have less than 1 test interactions
RP3betaRecommender: Similarity column 38121 (100.0%), 856.52 column/sec. Elapsed time 44.51 sec
SLIMElasticNetRecommender: Processed 8026 (21.1%) in 5.00 min. Items per second: 26.75
SLIMElasticNetRecommender: Processed 16575 (43.5%) in 10.00 min. Items per second: 27.62
SLIMElasticNetRecommender: Processed 25139 (65.9%) in 15.00 min. Items per second: 27.93
SLIMElasticNetRecommender: Processed 33388 (87.6%) in 20.00 min. Items per second: 27.82
SLIMElasticNetRecommender: Processed 38121 (100.0%) in 22.69 min. Items per second: 28.00
EvaluatorHoldout: Processed 35288 (100.0%) in 44.45 sec. Users per second: 794


[I 2024-12-10 14:21:22,295] Trial 0 finished with value: 0.025134236693618812 and parameters: {'alpha_rp3': 0.2527028750482182, 'beta_rp3': 0.6529141411137664, 'topK_rp3': 418, 'alpha_slim': 0.00222659963231494, 'l1_ratio_slim': 0.29442865165499427, 'topK_slim': 913, 'hybrid_alpha': 0.29815665970046457}. Best is trial 0 with value: 0.025134236693618812.


RP3betaRecommender: Similarity column 38121 (100.0%), 642.04 column/sec. Elapsed time 59.37 sec
SLIMElasticNetRecommender: Processed 5652 (14.8%) in 5.00 min. Items per second: 18.83
SLIMElasticNetRecommender: Processed 11134 (29.2%) in 10.00 min. Items per second: 18.55
SLIMElasticNetRecommender: Processed 17296 (45.4%) in 15.00 min. Items per second: 19.21
SLIMElasticNetRecommender: Processed 23749 (62.3%) in 20.00 min. Items per second: 19.79
SLIMElasticNetRecommender: Processed 29516 (77.4%) in 25.00 min. Items per second: 19.67
SLIMElasticNetRecommender: Processed 35967 (94.3%) in 30.00 min. Items per second: 19.98
SLIMElasticNetRecommender: Processed 38121 (100.0%) in 31.79 min. Items per second: 19.99
EvaluatorHoldout: Processed 35288 (100.0%) in 46.46 sec. Users per second: 760


[I 2024-12-10 14:55:34,022] Trial 1 finished with value: 0.02827115674917959 and parameters: {'alpha_rp3': 0.46444698025544484, 'beta_rp3': 0.4860494265097798, 'topK_rp3': 695, 'alpha_slim': 0.0007867315883947753, 'l1_ratio_slim': 0.20716834397465403, 'topK_slim': 979, 'hybrid_alpha': 0.8632640258592098}. Best is trial 1 with value: 0.02827115674917959.


...

RP3betaRecommender: Similarity column 38121 (100.0%), 575.29 column/sec. Elapsed time 1.10 min
SLIMElasticNetRecommender: Processed 2263 ( 5.9%) in 5.00 min. Items per second: 7.54
SLIMElasticNetRecommender: Processed 4535 (11.9%) in 10.00 min. Items per second: 7.56
SLIMElasticNetRecommender: Processed 7069 (18.5%) in 15.00 min. Items per second: 7.85
SLIMElasticNetRecommender: Processed 9705 (25.5%) in 20.00 min. Items per second: 8.09
SLIMElasticNetRecommender: Processed 12361 (32.4%) in 25.00 min. Items per second: 8.24
SLIMElasticNetRecommender: Processed 15026 (39.4%) in 30.01 min. Items per second: 8.35
SLIMElasticNetRecommender: Processed 17728 (46.5%) in 35.01 min. Items per second: 8.44
SLIMElasticNetRecommender: Processed 20391 (53.5%) in 40.01 min. Items per second: 8.49
SLIMElasticNetRecommender: Processed 22972 (60.3%) in 45.01 min. Items per second: 8.51
SLIMElasticNetRecommender: Processed 25687 (67.4%) in 50.01 min. Items per second: 8.56
SLIMElasticNetRecommender: Pro

[I 2024-12-10 22:35:10,389] Trial 8 finished with value: 0.03226386482387 and parameters: {'alpha_rp3': 0.32631186582949284, 'beta_rp3': 0.5938063493552193, 'topK_rp3': 772, 'alpha_slim': 9.136070060614252e-05, 'l1_ratio_slim': 0.06691488743673453, 'topK_slim': 666, 'hybrid_alpha': 0.6720409744402086}. Best is trial 3 with value: 0.03355395781829578.


RP3betaRecommender: Similarity column 38121 (100.0%), 1278.02 column/sec. Elapsed time 29.83 sec
SLIMElasticNetRecommender: Processed 2143 ( 5.6%) in 5.00 min. Items per second: 7.14
SLIMElasticNetRecommender: Processed 4313 (11.3%) in 10.00 min. Items per second: 7.18
SLIMElasticNetRecommender: Processed 6709 (17.6%) in 15.00 min. Items per second: 7.45
SLIMElasticNetRecommender: Processed 9219 (24.2%) in 20.00 min. Items per second: 7.68
SLIMElasticNetRecommender: Processed 11714 (30.7%) in 25.00 min. Items per second: 7.81
SLIMElasticNetRecommender: Processed 14175 (37.2%) in 30.01 min. Items per second: 7.87
SLIMElasticNetRecommender: Processed 16687 (43.8%) in 35.01 min. Items per second: 7.94
SLIMElasticNetRecommender: Processed 19106 (50.1%) in 40.01 min. Items per second: 7.96
SLIMElasticNetRecommender: Processed 21653 (56.8%) in 45.01 min. Items per second: 8.02
SLIMElasticNetRecommender: Processed 24209 (63.5%) in 50.01 min. Items per second: 8.07
SLIMElasticNetRecommender: P

[I 2024-12-10 23:53:55,889] Trial 9 finished with value: 0.03178112506432242 and parameters: {'alpha_rp3': 0.8871115697744515, 'beta_rp3': 0.5258934950741396, 'topK_rp3': 223, 'alpha_slim': 1.109518306311597e-05, 'l1_ratio_slim': 0.17148121001840685, 'topK_slim': 298, 'hybrid_alpha': 0.2543097755669951}. Best is trial 3 with value: 0.03355395781829578.


Best hyperparameters: {'alpha_rp3': 0.8711331767355596, 'beta_rp3': 0.502333441080462, 'topK_rp3': 844, 'alpha_slim': 0.00016765482284101813, 'l1_ratio_slim': 0.2127066557588067, 'topK_slim': 904, 'hybrid_alpha': 0.17730519572952572}
Best MAP: 0.03355395781829578
RP3betaRecommender: Similarity column 38121 (100.0%), 505.23 column/sec. Elapsed time 1.26 min
SLIMElasticNetRecommender: Processed 1702 ( 4.5%) in 5.00 min. Items per second: 5.67
SLIMElasticNetRecommender: Processed 3580 ( 9.4%) in 10.00 min. Items per second: 5.96
SLIMElasticNetRecommender: Processed 5605 (14.7%) in 15.00 min. Items per second: 6.22
SLIMElasticNetRecommender: Processed 7377 (19.4%) in 20.01 min. Items per second: 6.15
SLIMElasticNetRecommender: Processed 9349 (24.5%) in 25.01 min. Items per second: 6.23
SLIMElasticNetRecommender: Processed 11299 (29.6%) in 30.01 min. Items per second: 6.27
SLIMElasticNetRecommender: Processed 13338 (35.0%) in 35.01 min. Items per second: 6.35
SLIMElasticNetRecommender: Proc

In [20]:

# Save results
output_folder = "C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\"
os.makedirs(output_folder, exist_ok=True)
output_file = os.path.join(output_folder, "sample_submission_optunawithouticm.csv")

with open(output_file, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['user_id', 'item_list'])
    for user_id in target_users['user_id']:
        recommendations = hybrid_recommender.recommend(user_id, cutoff=10)
        writer.writerow([user_id, " ".join(map(str, recommendations))])

print(f"Results saved to {output_file}")

Results saved to C:\Users\VOLKAN MAZLUM\Desktop\Proje\sample_submission_optunawithouticm.csv


In [2]:
import os
import csv
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
import optuna
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from Evaluation.Evaluator import EvaluatorHoldout
from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender
from Recommenders.KNN.ItemKNNCustomSimilarityRecommender import ItemKNNCustomSimilarityRecommender

# Load data
data_train = pd.read_csv("C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\data_train.csv")
target_users = pd.read_csv("C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\data_target_users_test.csv")

# Create sparse matrix
n_users = data_train['user_id'].nunique()
n_items = data_train['item_id'].nunique()

URM_all = csr_matrix((data_train['data'], 
                      (data_train['user_id'], data_train['item_id'])),
                     shape=(n_users, n_items))

# Split data
def split_data(URM_all, train_percentage=0.9):
    URM_train_validation, URM_test = split_train_in_two_percentage_global_sample(URM_all, train_percentage)
    URM_train, URM_validation = split_train_in_two_percentage_global_sample(URM_train_validation, train_percentage)
    return URM_train, URM_validation, URM_test

# Objective function for Optuna
def objective(trial):
    # Suggest hyperparameters for RP3beta
    alpha_rp3 = trial.suggest_float("alpha_rp3", 0.15, 0.6)
    beta_rp3 = trial.suggest_float("beta_rp3", 0.03, 0.6)
    topK_rp3 = trial.suggest_int("topK_rp3", 15, 50)

    # Suggest hyperparameters for SLIMElasticNet
    alpha_slim = trial.suggest_float("alpha_slim", 4e-5, 5e-3, log=True)
    l1_ratio_slim = trial.suggest_float("l1_ratio_slim", 0.06, 0.6)
    topK_slim = trial.suggest_int("topK_slim", 1800, 2800)

    # Suggest weight for hybrid alpha
    hybrid_alpha = trial.suggest_float("hybrid_alpha", 0.1, 0.8)

    # Train RP3beta
    rp3_recommender = RP3betaRecommender(URM_train)
    rp3_recommender.fit(alpha=alpha_rp3, beta=beta_rp3, topK=topK_rp3)

    # Train SLIMElasticNet
    slim_recommender = SLIMElasticNetRecommender(URM_train)
    slim_recommender.fit(alpha=alpha_slim, l1_ratio=l1_ratio_slim, topK=topK_slim)

    # Combine recommendations using hybrid alpha
    hybrid_similarity = ((1 - hybrid_alpha) * rp3_recommender.W_sparse 
                         + hybrid_alpha * slim_recommender.W_sparse)

    hybrid_recommender = ItemKNNCustomSimilarityRecommender(URM_train)
    hybrid_recommender.fit(hybrid_similarity)

    # Evaluate on validation set
    result_dict, _ = evaluator_validation.evaluateRecommender(hybrid_recommender)
    map_score = result_dict["MAP"].values[0]

    return map_score

# Main workflow
def main(URM_all, n_trials):
    global URM_train, URM_validation, evaluator_validation
    
    # Split data
    URM_train, URM_validation, URM_test = split_data(URM_all)
    
    # Evaluation object
    evaluator_validation = EvaluatorHoldout(URM_validation, cutoff_list=[10])
    
    # Optimize hyperparameters with Optuna
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)
    
    print("Best hyperparameters:", study.best_params)
    print("Best MAP:", study.best_value)
    
    # Retrain on full data with best parameters
    best_params = study.best_params
    
    rp3_recommender = RP3betaRecommender(URM_all)
    rp3_recommender.fit(alpha=best_params["alpha_rp3"], beta=best_params["beta_rp3"], topK=best_params["topK_rp3"])
    
    slim_recommender = SLIMElasticNetRecommender(URM_all)
    slim_recommender.fit(alpha=best_params["alpha_slim"], l1_ratio=best_params["l1_ratio_slim"], topK=best_params["topK_slim"])
    
    hybrid_similarity = ((1 - best_params["hybrid_alpha"]) * rp3_recommender.W_sparse 
                         + best_params["hybrid_alpha"] * slim_recommender.W_sparse)
                         
    hybrid_recommender = ItemKNNCustomSimilarityRecommender(URM_all)
    hybrid_recommender.fit(hybrid_similarity)
    
    return hybrid_recommender, study


In [3]:

# Run the pipeline
n_trials = 16  # Number of Optuna trials
hybrid_recommender, study = main(URM_all, n_trials)



[I 2024-12-11 11:35:34,253] A new study created in memory with name: no-name-220d0119-6f9f-4566-9c23-2906d67b133c


EvaluatorHoldout: Ignoring 2586 ( 7.2%) Users that have less than 1 test interactions
RP3betaRecommender: Similarity column 38121 (100.0%), 2080.41 column/sec. Elapsed time 18.32 sec
SLIMElasticNetRecommender: Processed 7228 (19.0%) in 5.00 min. Items per second: 24.09
SLIMElasticNetRecommender: Processed 13224 (34.7%) in 10.00 min. Items per second: 22.03
SLIMElasticNetRecommender: Processed 17581 (46.1%) in 15.00 min. Items per second: 19.53
SLIMElasticNetRecommender: Processed 24372 (63.9%) in 20.00 min. Items per second: 20.31
SLIMElasticNetRecommender: Processed 28691 (75.3%) in 25.00 min. Items per second: 19.12
SLIMElasticNetRecommender: Processed 32172 (84.4%) in 30.00 min. Items per second: 17.87
SLIMElasticNetRecommender: Processed 35684 (93.6%) in 35.00 min. Items per second: 16.99
SLIMElasticNetRecommender: Processed 38121 (100.0%) in 38.48 min. Items per second: 16.51
EvaluatorHoldout: Processed 33150 (100.0%) in 56.74 sec. Users per second: 584


[I 2024-12-11 12:15:19,758] Trial 0 finished with value: 0.022971523139169777 and parameters: {'alpha_rp3': 0.41263961666208127, 'beta_rp3': 0.15776063357362347, 'topK_rp3': 17, 'alpha_slim': 0.00342885590922876, 'l1_ratio_slim': 0.567415580911343, 'topK_slim': 2799, 'hybrid_alpha': 0.38599909626610573}. Best is trial 0 with value: 0.022971523139169777.


RP3betaRecommender: Similarity column 38121 (100.0%), 1033.76 column/sec. Elapsed time 36.88 sec
SLIMElasticNetRecommender: Processed 2328 ( 6.1%) in 5.00 min. Items per second: 7.76
SLIMElasticNetRecommender: Processed 4777 (12.5%) in 10.00 min. Items per second: 7.96
SLIMElasticNetRecommender: Processed 7062 (18.5%) in 15.00 min. Items per second: 7.84
SLIMElasticNetRecommender: Processed 9445 (24.8%) in 20.00 min. Items per second: 7.87
SLIMElasticNetRecommender: Processed 11817 (31.0%) in 25.00 min. Items per second: 7.88
SLIMElasticNetRecommender: Processed 15204 (39.9%) in 30.00 min. Items per second: 8.44
SLIMElasticNetRecommender: Processed 19970 (52.4%) in 35.00 min. Items per second: 9.51
SLIMElasticNetRecommender: Processed 24378 (63.9%) in 40.00 min. Items per second: 10.16
SLIMElasticNetRecommender: Processed 28479 (74.7%) in 45.01 min. Items per second: 10.55
SLIMElasticNetRecommender: Processed 33200 (87.1%) in 50.01 min. Items per second: 11.06
SLIMElasticNetRecommender

[I 2024-12-11 13:12:02,757] Trial 1 finished with value: 0.022830636357106568 and parameters: {'alpha_rp3': 0.21712429446245285, 'beta_rp3': 0.41071607066144966, 'topK_rp3': 24, 'alpha_slim': 0.002898172353907439, 'l1_ratio_slim': 0.07364544580409102, 'topK_slim': 2580, 'hybrid_alpha': 0.10981693886828191}. Best is trial 0 with value: 0.022971523139169777.


...

RP3betaRecommender: Similarity column 38121 (100.0%), 1916.45 column/sec. Elapsed time 19.89 sec
SLIMElasticNetRecommender: Processed 1992 ( 5.2%) in 5.00 min. Items per second: 6.63
SLIMElasticNetRecommender: Processed 4075 (10.7%) in 10.01 min. Items per second: 6.79
SLIMElasticNetRecommender: Processed 6334 (16.6%) in 15.01 min. Items per second: 7.03
SLIMElasticNetRecommender: Processed 8619 (22.6%) in 20.01 min. Items per second: 7.18
SLIMElasticNetRecommender: Processed 10713 (28.1%) in 25.01 min. Items per second: 7.14
SLIMElasticNetRecommender: Processed 12156 (31.9%) in 30.02 min. Items per second: 6.75
SLIMElasticNetRecommender: Processed 14368 (37.7%) in 35.02 min. Items per second: 6.84
SLIMElasticNetRecommender: Processed 16643 (43.7%) in 40.02 min. Items per second: 6.93
SLIMElasticNetRecommender: Processed 18874 (49.5%) in 45.02 min. Items per second: 6.99
SLIMElasticNetRecommender: Processed 21232 (55.7%) in 50.02 min. Items per second: 7.07
SLIMElasticNetRecommender: P

[I 2024-12-12 04:40:53,436] Trial 14 finished with value: 0.025593940482175302 and parameters: {'alpha_rp3': 0.5984025506111565, 'beta_rp3': 0.28487111805866616, 'topK_rp3': 35, 'alpha_slim': 0.00017182375641198056, 'l1_ratio_slim': 0.24955291331217314, 'topK_slim': 2512, 'hybrid_alpha': 0.6402200452333124}. Best is trial 13 with value: 0.025663442984030747.


RP3betaRecommender: Similarity column 38121 (100.0%), 1816.13 column/sec. Elapsed time 20.99 sec
SLIMElasticNetRecommender: Processed 4817 (12.6%) in 5.00 min. Items per second: 16.05
SLIMElasticNetRecommender: Processed 9455 (24.8%) in 10.00 min. Items per second: 15.75
SLIMElasticNetRecommender: Processed 14589 (38.3%) in 15.00 min. Items per second: 16.20
SLIMElasticNetRecommender: Processed 19829 (52.0%) in 20.00 min. Items per second: 16.52
SLIMElasticNetRecommender: Processed 24923 (65.4%) in 25.00 min. Items per second: 16.61
SLIMElasticNetRecommender: Processed 29665 (77.8%) in 30.00 min. Items per second: 16.48
SLIMElasticNetRecommender: Processed 35089 (92.0%) in 35.01 min. Items per second: 16.71
SLIMElasticNetRecommender: Processed 38121 (100.0%) in 37.94 min. Items per second: 16.75
EvaluatorHoldout: Processed 33150 (100.0%) in 31.58 sec. Users per second: 1050


[I 2024-12-12 05:19:43,918] Trial 15 finished with value: 0.022267050922933015 and parameters: {'alpha_rp3': 0.33207103489675466, 'beta_rp3': 0.032043238938646745, 'topK_rp3': 22, 'alpha_slim': 0.0010122929807770502, 'l1_ratio_slim': 0.2988171792210811, 'topK_slim': 2224, 'hybrid_alpha': 0.6184177730047303}. Best is trial 13 with value: 0.025663442984030747.


Best hyperparameters: {'alpha_rp3': 0.4529010657038516, 'beta_rp3': 0.2647796370741364, 'topK_rp3': 28, 'alpha_slim': 0.00019739600530621637, 'l1_ratio_slim': 0.2126461775255941, 'topK_slim': 2452, 'hybrid_alpha': 0.6598271260641828}
Best MAP: 0.025663442984030747
RP3betaRecommender: Similarity column 38121 (100.0%), 1666.64 column/sec. Elapsed time 22.87 sec
SLIMElasticNetRecommender: Processed 1743 ( 4.6%) in 5.00 min. Items per second: 5.81
SLIMElasticNetRecommender: Processed 3613 ( 9.5%) in 10.00 min. Items per second: 6.02
SLIMElasticNetRecommender: Processed 5511 (14.5%) in 15.01 min. Items per second: 6.12
SLIMElasticNetRecommender: Processed 7345 (19.3%) in 20.01 min. Items per second: 6.12
SLIMElasticNetRecommender: Processed 9207 (24.2%) in 25.01 min. Items per second: 6.13
SLIMElasticNetRecommender: Processed 11078 (29.1%) in 30.01 min. Items per second: 6.15
SLIMElasticNetRecommender: Processed 12986 (34.1%) in 35.02 min. Items per second: 6.18
SLIMElasticNetRecommender: P

In [4]:

# Save results
output_folder = "C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\"
os.makedirs(output_folder, exist_ok=True)
output_file = os.path.join(output_folder, "sample_submission_optuna.csv")

with open(output_file, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['user_id', 'item_list'])
    for user_id in target_users['user_id']:
        recommendations = hybrid_recommender.recommend(user_id, cutoff=10)
        writer.writerow([user_id, " ".join(map(str, recommendations))])

print(f"Results saved to {output_file}")

Results saved to C:\Users\VOLKAN MAZLUM\Desktop\Proje\sample_submission_optuna.csv


In [2]:
import os
import csv
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
import optuna
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from Evaluation.Evaluator import EvaluatorHoldout
from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender
from Recommenders.KNN.ItemKNNCustomSimilarityRecommender import ItemKNNCustomSimilarityRecommender

# Load data
data_train = pd.read_csv("C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\data_train.csv")
target_users = pd.read_csv("C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\data_target_users_test.csv")

# Create sparse matrix
n_users = data_train['user_id'].nunique()
n_items = data_train['item_id'].nunique()

URM_all = csr_matrix((data_train['data'], 
                      (data_train['user_id'], data_train['item_id'])),
                     shape=(n_users, n_items))

# Split data
def split_data(URM_all, train_percentage=0.8):
    URM_train_validation, URM_test = split_train_in_two_percentage_global_sample(URM_all, train_percentage)
    URM_train, URM_validation = split_train_in_two_percentage_global_sample(URM_train_validation, train_percentage)
    return URM_train, URM_validation, URM_test

# Objective function for Optuna
def objective(trial):
    # Suggest hyperparameters for RP3beta
    alpha_rp3 = trial.suggest_float("alpha_rp3", 0.05, 0.9)
    beta_rp3 = trial.suggest_float("beta_rp3", 0.01, 0.9)
    topK_rp3 = trial.suggest_int("topK_rp3", 10, 75)

    # Suggest hyperparameters for SLIMElasticNet
    alpha_slim = trial.suggest_float("alpha_slim", 1e-5, 7e-3, log=True)
    l1_ratio_slim = trial.suggest_float("l1_ratio_slim", 0.01, 0.9)
    topK_slim = trial.suggest_int("topK_slim", 1500, 3500)

    # Suggest weight for hybrid alpha
    hybrid_alpha = trial.suggest_float("hybrid_alpha", 0.1, 0.9)

    # Train RP3beta
    rp3_recommender = RP3betaRecommender(URM_train)
    rp3_recommender.fit(alpha=alpha_rp3, beta=beta_rp3, topK=topK_rp3)

    # Train SLIMElasticNet
    slim_recommender = SLIMElasticNetRecommender(URM_train)
    slim_recommender.fit(alpha=alpha_slim, l1_ratio=l1_ratio_slim, topK=topK_slim)

    # Combine recommendations using hybrid alpha
    hybrid_similarity = ((1 - hybrid_alpha) * rp3_recommender.W_sparse 
                         + hybrid_alpha * slim_recommender.W_sparse)

    hybrid_recommender = ItemKNNCustomSimilarityRecommender(URM_train)
    hybrid_recommender.fit(hybrid_similarity)

    # Evaluate on validation set
    result_dict, _ = evaluator_validation.evaluateRecommender(hybrid_recommender)
    map_score = result_dict["MAP"].values[0]

    return map_score

# Main workflow
def main(URM_all, n_trials):
    global URM_train, URM_validation, evaluator_validation
    
    # Split data
    URM_train, URM_validation, URM_test = split_data(URM_all)
    
    # Evaluation object
    evaluator_validation = EvaluatorHoldout(URM_validation, cutoff_list=[10])
    
    # Optimize hyperparameters with Optuna
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)
    
    print("Best hyperparameters:", study.best_params)
    print("Best MAP:", study.best_value)
    
    # Retrain on full data with best parameters
    best_params = study.best_params
    
    rp3_recommender = RP3betaRecommender(URM_all)
    rp3_recommender.fit(alpha=best_params["alpha_rp3"], beta=best_params["beta_rp3"], topK=best_params["topK_rp3"])
    
    slim_recommender = SLIMElasticNetRecommender(URM_all)
    slim_recommender.fit(alpha=best_params["alpha_slim"], l1_ratio=best_params["l1_ratio_slim"], topK=best_params["topK_slim"])
    
    hybrid_similarity = ((1 - best_params["hybrid_alpha"]) * rp3_recommender.W_sparse 
                         + best_params["hybrid_alpha"] * slim_recommender.W_sparse)
                         
    hybrid_recommender = ItemKNNCustomSimilarityRecommender(URM_all)
    hybrid_recommender.fit(hybrid_similarity)
    
    return hybrid_recommender, study


In [3]:

# Run the pipeline
n_trials = 27  # Number of Optuna trials
hybrid_recommender, study = main(URM_all, n_trials)



[I 2024-12-13 02:03:35,992] A new study created in memory with name: no-name-469355ba-19f8-4b24-8458-4b5d9ab0fd36


EvaluatorHoldout: Ignoring 2538 ( 7.1%) Users that have less than 1 test interactions
RP3betaRecommender: Similarity column 38121 (100.0%), 1787.47 column/sec. Elapsed time 21.33 sec
SLIMElasticNetRecommender: Processed 1982 ( 5.2%) in 5.00 min. Items per second: 6.60
SLIMElasticNetRecommender: Processed 3728 ( 9.8%) in 10.00 min. Items per second: 6.21
SLIMElasticNetRecommender: Processed 5723 (15.0%) in 15.00 min. Items per second: 6.36
SLIMElasticNetRecommender: Processed 7906 (20.7%) in 20.00 min. Items per second: 6.59
SLIMElasticNetRecommender: Processed 10029 (26.3%) in 25.01 min. Items per second: 6.68
SLIMElasticNetRecommender: Processed 12079 (31.7%) in 30.01 min. Items per second: 6.71
SLIMElasticNetRecommender: Processed 14160 (37.1%) in 35.01 min. Items per second: 6.74
SLIMElasticNetRecommender: Processed 16176 (42.4%) in 40.01 min. Items per second: 6.74
SLIMElasticNetRecommender: Processed 18207 (47.8%) in 45.01 min. Items per second: 6.74
SLIMElasticNetRecommender: Pro

[I 2024-12-13 03:34:56,297] Trial 0 finished with value: 0.02489921581621325 and parameters: {'alpha_rp3': 0.40275129662742143, 'beta_rp3': 0.3044021167285963, 'topK_rp3': 42, 'alpha_slim': 1.773906351266048e-05, 'l1_ratio_slim': 0.3964534892396956, 'topK_slim': 3117, 'hybrid_alpha': 0.590638475024499}. Best is trial 0 with value: 0.02489921581621325.


RP3betaRecommender: Similarity column 38121 (100.0%), 1722.91 column/sec. Elapsed time 22.13 sec
SLIMElasticNetRecommender: Processed 6546 (17.2%) in 5.00 min. Items per second: 21.81
SLIMElasticNetRecommender: Processed 13000 (34.1%) in 10.00 min. Items per second: 21.66
SLIMElasticNetRecommender: Processed 20556 (53.9%) in 15.00 min. Items per second: 22.84
SLIMElasticNetRecommender: Processed 27224 (71.4%) in 20.00 min. Items per second: 22.68
SLIMElasticNetRecommender: Processed 34026 (89.3%) in 25.00 min. Items per second: 22.68
SLIMElasticNetRecommender: Processed 38121 (100.0%) in 28.03 min. Items per second: 22.66
EvaluatorHoldout: Processed 33198 (100.0%) in 31.75 sec. Users per second: 1046


[I 2024-12-13 04:03:56,190] Trial 1 finished with value: 0.02070680653931137 and parameters: {'alpha_rp3': 0.24658905741232773, 'beta_rp3': 0.07942512546970273, 'topK_rp3': 71, 'alpha_slim': 0.006950735149479435, 'l1_ratio_slim': 0.7558031780168536, 'topK_slim': 1546, 'hybrid_alpha': 0.7193798075633231}. Best is trial 0 with value: 0.02489921581621325.


...

RP3betaRecommender: Similarity column 38121 (100.0%), 1978.34 column/sec. Elapsed time 19.27 sec
SLIMElasticNetRecommender: Processed 1913 ( 5.0%) in 5.00 min. Items per second: 6.37
SLIMElasticNetRecommender: Processed 3821 (10.0%) in 10.00 min. Items per second: 6.36
SLIMElasticNetRecommender: Processed 5704 (15.0%) in 15.01 min. Items per second: 6.33
SLIMElasticNetRecommender: Processed 7934 (20.8%) in 20.01 min. Items per second: 6.61
SLIMElasticNetRecommender: Processed 10183 (26.7%) in 25.01 min. Items per second: 6.78
SLIMElasticNetRecommender: Processed 12423 (32.6%) in 30.01 min. Items per second: 6.90
SLIMElasticNetRecommender: Processed 14726 (38.6%) in 35.01 min. Items per second: 7.01
SLIMElasticNetRecommender: Processed 16965 (44.5%) in 40.01 min. Items per second: 7.07
SLIMElasticNetRecommender: Processed 19216 (50.4%) in 45.01 min. Items per second: 7.11
SLIMElasticNetRecommender: Processed 21522 (56.5%) in 50.01 min. Items per second: 7.17
SLIMElasticNetRecommender: P

[I 2024-12-14 10:26:28,544] Trial 25 finished with value: 0.02505176612283918 and parameters: {'alpha_rp3': 0.057407423285867995, 'beta_rp3': 0.3722039564883967, 'topK_rp3': 16, 'alpha_slim': 3.91024425890771e-05, 'l1_ratio_slim': 0.3701582599725991, 'topK_slim': 1734, 'hybrid_alpha': 0.6986116984727613}. Best is trial 22 with value: 0.025275647701094514.


RP3betaRecommender: Similarity column 38121 (100.0%), 2087.32 column/sec. Elapsed time 18.26 sec
SLIMElasticNetRecommender: Processed 2777 ( 7.3%) in 5.00 min. Items per second: 9.25
SLIMElasticNetRecommender: Processed 5794 (15.2%) in 10.00 min. Items per second: 9.65
SLIMElasticNetRecommender: Processed 8679 (22.8%) in 15.00 min. Items per second: 9.64
SLIMElasticNetRecommender: Processed 11617 (30.5%) in 20.00 min. Items per second: 9.68
SLIMElasticNetRecommender: Processed 14831 (38.9%) in 25.00 min. Items per second: 9.89
SLIMElasticNetRecommender: Processed 17849 (46.8%) in 30.01 min. Items per second: 9.91
SLIMElasticNetRecommender: Processed 20998 (55.1%) in 35.01 min. Items per second: 10.00
SLIMElasticNetRecommender: Processed 24136 (63.3%) in 40.01 min. Items per second: 10.05
SLIMElasticNetRecommender: Processed 27303 (71.6%) in 45.01 min. Items per second: 10.11
SLIMElasticNetRecommender: Processed 30412 (79.8%) in 50.01 min. Items per second: 10.14
SLIMElasticNetRecommend

[I 2024-12-14 11:29:18,700] Trial 26 finished with value: 0.024941126449575347 and parameters: {'alpha_rp3': 0.28755378262567505, 'beta_rp3': 0.27479424281202086, 'topK_rp3': 24, 'alpha_slim': 0.00023834872601586563, 'l1_ratio_slim': 0.4107192309031346, 'topK_slim': 2629, 'hybrid_alpha': 0.6297661489039059}. Best is trial 22 with value: 0.025275647701094514.


Best hyperparameters: {'alpha_rp3': 0.13622088798736692, 'beta_rp3': 0.3848020049980613, 'topK_rp3': 18, 'alpha_slim': 0.00010727757388842865, 'l1_ratio_slim': 0.30172517043157504, 'topK_slim': 1796, 'hybrid_alpha': 0.6896866280565166}
Best MAP: 0.025275647701094514
RP3betaRecommender: Similarity column 38121 (100.0%), 1650.41 column/sec. Elapsed time 23.10 sec
SLIMElasticNetRecommender: Processed 1746 ( 4.6%) in 5.00 min. Items per second: 5.82
SLIMElasticNetRecommender: Processed 3442 ( 9.0%) in 10.00 min. Items per second: 5.73
SLIMElasticNetRecommender: Processed 5167 (13.6%) in 15.00 min. Items per second: 5.74
SLIMElasticNetRecommender: Processed 7179 (18.8%) in 20.01 min. Items per second: 5.98
SLIMElasticNetRecommender: Processed 9160 (24.0%) in 25.01 min. Items per second: 6.10
SLIMElasticNetRecommender: Processed 11102 (29.1%) in 30.01 min. Items per second: 6.16
SLIMElasticNetRecommender: Processed 13024 (34.2%) in 35.01 min. Items per second: 6.20
SLIMElasticNetRecommender:

In [4]:

# Save results
output_folder = "C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\"
os.makedirs(output_folder, exist_ok=True)
output_file = os.path.join(output_folder, "sample_submission_optuna.csv")

with open(output_file, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['user_id', 'item_list'])
    for user_id in target_users['user_id']:
        recommendations = hybrid_recommender.recommend(user_id, cutoff=10)
        writer.writerow([user_id, " ".join(map(str, recommendations))])

print(f"Results saved to {output_file}")

Results saved to C:\Users\VOLKAN MAZLUM\Desktop\Proje\sample_submission_optuna.csv


In [2]:
import os
import csv
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
import optuna
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from Evaluation.Evaluator import EvaluatorHoldout
from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender
from Recommenders.KNN.ItemKNNCustomSimilarityRecommender import ItemKNNCustomSimilarityRecommender

# Load data
data_train = pd.read_csv("C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\data_train.csv")
target_users = pd.read_csv("C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\data_target_users_test.csv")

# Create sparse matrix
n_users = data_train['user_id'].nunique()
n_items = data_train['item_id'].nunique()

URM_all = csr_matrix((data_train['data'], 
                      (data_train['user_id'], data_train['item_id'])),
                     shape=(n_users, n_items))

# Split data
def split_data(URM_all, train_percentage=0.8):
    URM_train_validation, URM_test = split_train_in_two_percentage_global_sample(URM_all, train_percentage)
    URM_train, URM_validation = split_train_in_two_percentage_global_sample(URM_train_validation, train_percentage)
    return URM_train, URM_validation, URM_test

# Objective function for Optuna
def objective(trial):
    # Suggest hyperparameters for RP3beta
    alpha_rp3 = trial.suggest_float("alpha_rp3", 0.05, 0.9)
    beta_rp3 = trial.suggest_float("beta_rp3", 0.02, 0.9)
    topK_rp3 = trial.suggest_int("topK_rp3", 15, 75)

    # Suggest hyperparameters for SLIMElasticNet
    alpha_slim = trial.suggest_float("alpha_slim", 1e-5, 7e-3, log=True)
    l1_ratio_slim = trial.suggest_float("l1_ratio_slim", 0.01, 0.9)
    topK_slim = trial.suggest_int("topK_slim", 1700, 3500)

    # Suggest weight for hybrid alpha
    hybrid_alpha = trial.suggest_float("hybrid_alpha", 0.1, 0.9)

    # Train RP3beta
    rp3_recommender = RP3betaRecommender(URM_train)
    rp3_recommender.fit(alpha=alpha_rp3, beta=beta_rp3, topK=topK_rp3)

    # Train SLIMElasticNet
    slim_recommender = SLIMElasticNetRecommender(URM_train)
    slim_recommender.fit(alpha=alpha_slim, l1_ratio=l1_ratio_slim, topK=topK_slim)

    # Combine recommendations using hybrid alpha
    hybrid_similarity = ((1 - hybrid_alpha) * rp3_recommender.W_sparse 
                         + hybrid_alpha * slim_recommender.W_sparse)

    hybrid_recommender = ItemKNNCustomSimilarityRecommender(URM_train)
    hybrid_recommender.fit(hybrid_similarity)

    # Evaluate on validation set
    result_dict, _ = evaluator_validation.evaluateRecommender(hybrid_recommender)
    map_score = result_dict["MAP"].values[0]

    return map_score

# Main workflow
def main(URM_all, n_trials):
    global URM_train, URM_validation, evaluator_validation
    
    # Split data
    URM_train, URM_validation, URM_test = split_data(URM_all)
    
    # Evaluation object
    evaluator_validation = EvaluatorHoldout(URM_validation, cutoff_list=[10])
    
    # Optimize hyperparameters with Optuna
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)
    
    print("Best hyperparameters:", study.best_params)
    print("Best MAP:", study.best_value)
    
    # Retrain on full data with best parameters
    best_params = study.best_params
    
    rp3_recommender = RP3betaRecommender(URM_all)
    rp3_recommender.fit(alpha=best_params["alpha_rp3"], beta=best_params["beta_rp3"], topK=best_params["topK_rp3"])
    
    slim_recommender = SLIMElasticNetRecommender(URM_all)
    slim_recommender.fit(alpha=best_params["alpha_slim"], l1_ratio=best_params["l1_ratio_slim"], topK=best_params["topK_slim"])
    
    hybrid_similarity = ((1 - best_params["hybrid_alpha"]) * rp3_recommender.W_sparse 
                         + best_params["hybrid_alpha"] * slim_recommender.W_sparse)
                         
    hybrid_recommender = ItemKNNCustomSimilarityRecommender(URM_all)
    hybrid_recommender.fit(hybrid_similarity)
    
    return hybrid_recommender, study


In [3]:

# Run the pipeline
n_trials = 50  # Number of Optuna trials
hybrid_recommender, study = main(URM_all, n_trials)



[I 2024-12-15 13:24:49,484] A new study created in memory with name: no-name-8346ff89-7bdf-43df-8441-1cb6e80b4afa


EvaluatorHoldout: Ignoring 2534 ( 7.1%) Users that have less than 1 test interactions
RP3betaRecommender: Similarity column 38121 (100.0%), 2010.72 column/sec. Elapsed time 18.96 sec
SLIMElasticNetRecommender: Processed 6442 (16.9%) in 5.00 min. Items per second: 21.47
SLIMElasticNetRecommender: Processed 11542 (30.3%) in 10.00 min. Items per second: 19.23
SLIMElasticNetRecommender: Processed 16040 (42.1%) in 15.00 min. Items per second: 17.82
SLIMElasticNetRecommender: Processed 22710 (59.6%) in 20.00 min. Items per second: 18.92
SLIMElasticNetRecommender: Processed 29365 (77.0%) in 25.00 min. Items per second: 19.57
SLIMElasticNetRecommender: Processed 35955 (94.3%) in 30.00 min. Items per second: 19.97
SLIMElasticNetRecommender: Processed 38121 (100.0%) in 31.66 min. Items per second: 20.07
EvaluatorHoldout: Processed 33202 (100.0%) in 31.67 sec. Users per second: 1048


[I 2024-12-15 13:57:24,298] Trial 0 finished with value: 0.020903675959470586 and parameters: {'alpha_rp3': 0.7594936528247445, 'beta_rp3': 0.42382641163467283, 'topK_rp3': 55, 'alpha_slim': 0.00674734580206007, 'l1_ratio_slim': 0.49298807279476664, 'topK_slim': 3375, 'hybrid_alpha': 0.5221302038277055}. Best is trial 0 with value: 0.020903675959470586.


RP3betaRecommender: Similarity column 38121 (100.0%), 1831.44 column/sec. Elapsed time 20.81 sec
SLIMElasticNetRecommender: Processed 5923 (15.5%) in 5.00 min. Items per second: 19.74
SLIMElasticNetRecommender: Processed 11948 (31.3%) in 10.00 min. Items per second: 19.91
SLIMElasticNetRecommender: Processed 18207 (47.8%) in 15.00 min. Items per second: 20.23
SLIMElasticNetRecommender: Processed 24263 (63.6%) in 20.00 min. Items per second: 20.22
SLIMElasticNetRecommender: Processed 30408 (79.8%) in 25.00 min. Items per second: 20.27
SLIMElasticNetRecommender: Processed 36923 (96.9%) in 30.00 min. Items per second: 20.51
SLIMElasticNetRecommender: Processed 38121 (100.0%) in 30.91 min. Items per second: 20.55
EvaluatorHoldout: Processed 33202 (100.0%) in 29.74 sec. Users per second: 1116


[I 2024-12-15 14:29:12,311] Trial 1 finished with value: 0.022729739698220175 and parameters: {'alpha_rp3': 0.1943282392363178, 'beta_rp3': 0.1075750905547798, 'topK_rp3': 27, 'alpha_slim': 0.0010482749711310966, 'l1_ratio_slim': 0.5879492099214572, 'topK_slim': 2890, 'hybrid_alpha': 0.15407332720315603}. Best is trial 1 with value: 0.022729739698220175.


...

RP3betaRecommender: Similarity column 38121 (100.0%), 1530.44 column/sec. Elapsed time 24.91 sec
SLIMElasticNetRecommender: Processed 2165 ( 5.7%) in 5.00 min. Items per second: 7.21
SLIMElasticNetRecommender: Processed 4721 (12.4%) in 10.00 min. Items per second: 7.86
SLIMElasticNetRecommender: Processed 7231 (19.0%) in 15.00 min. Items per second: 8.03
SLIMElasticNetRecommender: Processed 9703 (25.5%) in 20.00 min. Items per second: 8.08
SLIMElasticNetRecommender: Processed 12205 (32.0%) in 25.01 min. Items per second: 8.13
SLIMElasticNetRecommender: Processed 14814 (38.9%) in 30.01 min. Items per second: 8.23
SLIMElasticNetRecommender: Processed 17356 (45.5%) in 35.01 min. Items per second: 8.26
SLIMElasticNetRecommender: Processed 19987 (52.4%) in 40.01 min. Items per second: 8.33
SLIMElasticNetRecommender: Processed 22618 (59.3%) in 45.01 min. Items per second: 8.38
SLIMElasticNetRecommender: Processed 25251 (66.2%) in 50.01 min. Items per second: 8.42
SLIMElasticNetRecommender: P

[I 2024-12-18 04:29:20,296] Trial 48 finished with value: 0.025115826594878637 and parameters: {'alpha_rp3': 0.2777194699971725, 'beta_rp3': 0.4792005264992286, 'topK_rp3': 67, 'alpha_slim': 0.00021528092656346258, 'l1_ratio_slim': 0.2970106813900369, 'topK_slim': 2843, 'hybrid_alpha': 0.8012276181748401}. Best is trial 25 with value: 0.025456373110053117.


RP3betaRecommender: Similarity column 38121 (100.0%), 1728.57 column/sec. Elapsed time 22.05 sec
SLIMElasticNetRecommender: Processed 1967 ( 5.2%) in 5.00 min. Items per second: 6.55
SLIMElasticNetRecommender: Processed 4179 (11.0%) in 10.00 min. Items per second: 6.96
SLIMElasticNetRecommender: Processed 6350 (16.7%) in 15.00 min. Items per second: 7.05
SLIMElasticNetRecommender: Processed 8434 (22.1%) in 20.00 min. Items per second: 7.03
SLIMElasticNetRecommender: Processed 10688 (28.0%) in 25.00 min. Items per second: 7.12
SLIMElasticNetRecommender: Processed 12913 (33.9%) in 30.00 min. Items per second: 7.17
SLIMElasticNetRecommender: Processed 15102 (39.6%) in 35.00 min. Items per second: 7.19
SLIMElasticNetRecommender: Processed 17313 (45.4%) in 40.01 min. Items per second: 7.21
SLIMElasticNetRecommender: Processed 19480 (51.1%) in 45.01 min. Items per second: 7.21
SLIMElasticNetRecommender: Processed 20968 (55.0%) in 50.01 min. Items per second: 6.99
SLIMElasticNetRecommender: P

[I 2024-12-18 05:59:20,120] Trial 49 finished with value: 0.025325949718079267 and parameters: {'alpha_rp3': 0.2234322840602784, 'beta_rp3': 0.5552442005340805, 'topK_rp3': 18, 'alpha_slim': 2.7335841271697482e-05, 'l1_ratio_slim': 0.8276782700131365, 'topK_slim': 2285, 'hybrid_alpha': 0.5744928300989085}. Best is trial 25 with value: 0.025456373110053117.


Best hyperparameters: {'alpha_rp3': 0.5162025400481456, 'beta_rp3': 0.455342307582696, 'topK_rp3': 31, 'alpha_slim': 4.8976371168110366e-05, 'l1_ratio_slim': 0.6444925443894357, 'topK_slim': 2830, 'hybrid_alpha': 0.6012556906777773}
Best MAP: 0.025456373110053117
RP3betaRecommender: Similarity column 38121 (100.0%), 1655.25 column/sec. Elapsed time 23.03 sec
SLIMElasticNetRecommender: Processed 1696 ( 4.4%) in 5.00 min. Items per second: 5.65
SLIMElasticNetRecommender: Processed 3557 ( 9.3%) in 10.00 min. Items per second: 5.93
SLIMElasticNetRecommender: Processed 5383 (14.1%) in 15.00 min. Items per second: 5.98
SLIMElasticNetRecommender: Processed 7216 (18.9%) in 20.00 min. Items per second: 6.01
SLIMElasticNetRecommender: Processed 9062 (23.8%) in 25.00 min. Items per second: 6.04
SLIMElasticNetRecommender: Processed 10928 (28.7%) in 30.00 min. Items per second: 6.07
SLIMElasticNetRecommender: Processed 12782 (33.5%) in 35.00 min. Items per second: 6.09
SLIMElasticNetRecommender: Pr

In [4]:

# Save results
output_folder = "C:\\Users\\VOLKAN MAZLUM\\Desktop\\Proje\\"
os.makedirs(output_folder, exist_ok=True)
output_file = os.path.join(output_folder, "sample_submission.csv")

with open(output_file, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['user_id', 'item_list'])
    for user_id in target_users['user_id']:
        recommendations = hybrid_recommender.recommend(user_id, cutoff=10)
        writer.writerow([user_id, " ".join(map(str, recommendations))])

print(f"Results saved to {output_file}")

Results saved to C:\Users\VOLKAN MAZLUM\Desktop\Proje\sample_submission.csv
